# CVBoost - Fine-tuning LoRA (partie 3)

Ce notebook tourne sur **Google Colab avec un GPU** (Execution > Modifier le type d'execution > GPU T4),
puis Execution > Tout executer.

Tout le code est dans les fichiers **`ai_engine/train_lora.py`** (entrainement) et
**`ai_engine/evaluate_model.py`** (score SacreBLEU et traitement par lot) du projet
(lisible dans VS Code et sur GitHub). Ce notebook se contente de :
1. installer les bibliotheques ;
2. recuperer le projet ;
3. lancer `python -m ai_engine.train_lora` puis `python -m ai_engine.evaluate_model` ;
4. telecharger les deux resultats a ajouter au depot.

Aucune cle ni aucun secret n'est necessaire : le depot est public.

In [ ]:
# Recupere le projet (branche de travail) et se place dedans
BRANCHE = 'claude/capstone-github-setup-g24r7p'
!git clone -q --depth 1 -b {BRANCHE} https://github.com/jason-bit-lang/cvboost_enterprise.git
%cd cvboost_enterprise

In [ ]:
# Bibliotheques du fine-tuning, versions fixees (PyTorch, pandas et numpy sont deja sur Colab)
!pip install -q -r requirements-ia.txt

## Entrainement, score SacreBLEU et traitement par lot

Les deux scripts :
- reverifie qu'aucune offre n'est partagee entre train, validation et test (empreintes SHA-256) ;
- charge `google/flan-t5-small`, gele ses poids et ajoute LoRA sur les matrices requete et valeur
  (`q` et `v` dans T5, appelees `q_proj` et `v_proj` dans d'autres modeles) ;
- `train_lora.py` entraine 20 epoques avec `Seq2SeqTrainer`, garde la meilleure en validation et enregistre
  l'adaptateur dans `ai_engine/lora_cvboost/` ;
- `evaluate_model.py` affiche le score SacreBLEU sur le jeu de test, avec et sans LoRA, et quelques exemples ;
- puis fait lire toutes les offres au modele et ecrit `data/offres_ia.csv` (scores dans `scores.json`).

In [ ]:
!python -m ai_engine.train_lora
!python -m ai_engine.evaluate_model

## Telechargement

Deux fichiers a ajouter au depot (ou a envoyer a Claude) :
- `offres_ia.csv` -> `data/offres_ia.csv`
- `lora_cvboost.zip` -> a decompresser dans `ai_engine/lora_cvboost/`

In [ ]:
!cp data/offres_ia.csv offres_ia.csv
!cd ai_engine && zip -q -r ../lora_cvboost.zip lora_cvboost
!cat ai_engine/lora_cvboost/scores.json

from google.colab import files
files.download('offres_ia.csv')
files.download('lora_cvboost.zip')